# 2 · Sensitivity analysis: which numbers actually matter?

### What this notebook is for (in one minute)

We built a computer model that estimates how much energy every home in a city uses.
The model contains a few dozen **parameters**: settings we measured from real data,
like "how much extra gas a home burns for each degree the weather drops." No
measurement is perfect, so a fair question is: **if one of those settings were a bit
wrong, would the model's answer change a little, or a lot?**

That is what a **sensitivity analysis** does. Think of a recipe: some ingredients you
can eyeball (a pinch more salt won't ruin the dish), but others must be exact (double
the baking soda and it's inedible). This notebook finds the "baking soda" parameters,
the ones whose accuracy actually decides the answer, so we know where to focus our
scrutiny, and so a reader knows how much to trust the model.

### What the model is (plain version)

It is an **agent-based model**: the computer creates a stand-in for every real dwelling
in the city (its size, age, heating fuel, how many people live there) and simulates each
one's energy use **hour by hour for a whole year** (8,760 hours), driven by the real
weather. Add up all the homes and you get the city's demand. The settings that drive each
home were **calibrated** in notebook 1, tuned to match a national smart-meter dataset.
This notebook stress-tests those settings.

### How this notebook is organised (you don't need to code to follow it)

Each numbered **Step** answers one question in plain language first, then shows the code
and the result. You can read only the text and the charts and get the whole story.

| Step | The question it answers |
|---|---|
| 0 | Does the model's internal accounting add up? (a sanity check before we trust anything) |
| 1 | How big is each *piece* of demand: heating vs hot water vs appliances vs occupants? |
| 2 | Which parameters, if wrong, would move the total the most? (the headline result) |
| 3 | Is our fast shortcut for Step 2 actually correct? (we double-check it against the full model) |
| 4 | Why do some parameters *not* matter for the total? |
| 5 | For a switch-to-heat-pumps scenario, which single number matters most? |
| 6 | A second, independent method (weighting by how *precisely* each number was measured) that proves the model is well-behaved and adds what Step 2 leaves out. |

### How we compute influence (the one bit of method)

Total demand is just a sum of a few pieces (a home's always-on **baseline**, the extra
from **people**, and the **heating**). Most parameters feed one piece and do so
proportionally: double the heating sensitivity and the heating piece doubles. That means
we can run the full model **once** to measure how big each piece is, then work out every
parameter's influence with arithmetic instead of re-running the model hundreds of times.
We then **prove that shortcut is exact** by re-running the real model in Steps 3 and 6.

> **⚙️ Assumptions are flagged like this** throughout. An *assumption* is a number we
> chose or borrowed from the literature rather than measured from the smart-meter data.
> Those deserve extra scrutiny, so we call each one out where it appears.

⏱️ **Runtime:** one full-year run on a small sample of neighbourhoods (a few minutes),
two short check-runs, and, for Step 6, a heavier screen that is pre-computed and cached.

### Key terms (plain-language glossary)

Skip this if you know the jargon; refer back if a word is unfamiliar.

| Term | Plain meaning |
|---|---|
| **Parameter** | A setting inside the model (e.g. how much heat a home loses per cold degree). Sensitivity analysis asks how much each one matters. |
| **Calibration** | Tuning those settings so the model matches real measured data. Done in notebook 1. |
| **SERL** | The *Smart Energy Research Lab*, a national panel of ~13,000 UK homes with smart meters. The data we calibrated to. |
| **DESNZ** | The UK government department whose independent neighbourhood energy statistics we validate against in notebook 3. |
| **LSOA** | *Lower-layer Super Output Area*, a small census neighbourhood of ~650 households. The model reports demand per LSOA. |
| **Baseline / standing load** | A home's always-on electricity and its cooking/hot-water gas: everything that isn't space heating. |
| **Anchor** | The average size of a baseline (e.g. the typical home's standing electricity per hour). |
| **Heating slope** | How fast heating demand rises as it gets colder, in kWh per "degree-hour" of cold. |
| **Setpoint** | The outdoor temperature below which homes start heating (the model's ~15.8 °C "balance point"). |
| **Multiplier** | A per-home adjustment (bigger/older/less-efficient homes use more or less). Built to *average to 1.0* across the stock, so they shift demand *between* homes without changing the *total*. |
| **COP** | *Coefficient of Performance* of a heat pump: units of heat delivered per unit of electricity (≈2.8 here). Only relevant when converting homes to heat pumps. |
| **Perturbation** | Deliberately nudging one parameter (say +10%) to see how much the answer moves. |
| **Tornado chart** | A bar chart ranking parameters by influence, longest bar on top; it looks like a tornado. |
| **Standard error (SE)** | The measurement uncertainty on a calibrated number: how far off it could plausibly be, from the data. |
| **Linear / near-linear** | The output moves in proportion to the input, with no surprises, which is what lets us use the arithmetic shortcut. |
| **Morris screen (μ\*, σ)** | A standard "global" sensitivity method (Step 6). **μ\*** = a parameter's overall influence; **σ** = how much that influence changes depending on the other settings (its interaction/non-linearity). Small σ means the simple picture holds. |

In [ ]:
import sys, os, tempfile
from pathlib import Path
import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
import geopandas as gpd

REPO = Path.cwd()
while not (REPO / "household_energy").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)
SCRIPTS = REPO / "research" / "applied" / "scripts"
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(SCRIPTS))
import importlib, fit_serl_ledger as L
importlib.reload(L)

CONFIG = REPO / "household_energy" / "calibrated_config.yaml"
cfg = yaml.safe_load(CONFIG.read_text())["model"]
YEAR = 2023; SAMPLE_LSOAS = 15
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})
print("sensitivity of config:", CONFIG.relative_to(REPO))

## Step 0 · does the model's internal accounting add up?

**In plain terms:** before trusting any breakdown of demand, we check that the model's
books balance. The model records each home's energy in two different ways at once: once
split by *purpose* (baseline vs people vs heating), and once split by *fuel* (electricity
vs gas). Those two views describe the same energy, so they must add up to the same total,
exactly like double-entry bookkeeping, where the debits and credits have to match. If they
don't, a breakdown built on them would be meaningless, so we check first.

The cell below builds a **miniature model** (just two neighbourhoods), advances it by a
single simulated hour, prints the actual numbers for one gas-heated and one electric-heated
home so you can see the pieces, and then confirms the two views match across every home.
Step 1 repeats the check over a full year. Only then do we rely on the breakdown.

Every result below reads five per-home attributes the engine updates **each
simulated hour** (see `household_energy/model.py::step` and
`agent.py::apply_climate`):

| attribute | what it is | where it's set each hour |
|---|---|---|
| `base_kwh` | the house standing load (anchor × multipliers × shapes) | model step, after `reset_energy()` |
| `spike_kwh` | the people term $(n_d-\bar n)\,\sigma_h$, **negative** for below-average households | `_apply_serl_person_loads` |
| `heat_kwh` | space heating (or cooling-hour 0) after all heating multipliers and reshaping | `apply_climate` + intraday reshaping |
| `electric_kwh`, `gas_kwh`, `other_kwh` | the same energy, routed by fuel: baseline split by fuel, people → electricity, heating → the home's heating fuel | all of the above |

If the bookkeeping is right, then for every home, every hour:
$$\texttt{electric} + \texttt{gas} + \texttt{other} \;=\; \texttt{base} + \texttt{spike} + \texttt{heat} + \texttt{cool}$$
The cell below builds a tiny model, steps it one hour, shows one gas-heated and one
electric-heated home's actual numbers, and checks the identity across every home.
Step 1 then re-checks it over the full year. Only *then* do we trust the
decomposition.

In [ ]:
from household_energy.model import EnergyModel
GEO = "data/epc_abm_newcastle.geojson"; CLIM = "data/ncc_2t_timeseries_2010_2026.parquet"
gdf_all = gpd.read_file(GEO); gdf_all["lsoa_code"] = gdf_all["lsoa_code"].astype(str)

tiny = gdf_all[gdf_all.lsoa_code.isin(sorted(gdf_all.lsoa_code.unique())[:2])].copy()
mt = EnergyModel(gdf=tiny, climate_parquet=CLIM, climate_start=pd.Timestamp(f"{YEAR}-01-01", tz="UTC"),
                 local_tz="Europe/London", collect_agent_level=False, config_path=str(CONFIG))
mt.step()                                        # one simulated hour (1 Jan, midnight UTC)

def show(a, label):
    print(f"{label}  (fuel bucket = {a._resolve_heating_fuel_bucket()}, "
          f"{len(a.residents)} residents, {a.floor_area_m2:.0f} m²)")
    for attr, meaning in [("base_kwh", "house standing load"),
                          ("spike_kwh", "people (deviation from panel-mean occupancy)"),
                          ("heat_kwh", "space heating"),
                          ("electric_kwh", "-> billed to electricity"),
                          ("gas_kwh", "-> billed to gas")]:
        print(f"   {attr:13s} {getattr(a, attr, 0.0):+8.4f} kWh   {meaning}")

ag_t = mt.household_agents
show(next(a for a in ag_t if a._resolve_heating_fuel_bucket() == "gas"), "a gas-heated home, this hour:")
print()
show(next(a for a in ag_t if a._resolve_heating_fuel_bucket() == "electric"), "an electric-heated home, this hour:")

lhs = sum(getattr(a, "electric_kwh", 0.0) + getattr(a, "gas_kwh", 0.0) + getattr(a, "other_kwh", 0.0) for a in ag_t)
rhs = sum(getattr(a, "base_kwh", 0.0) + getattr(a, "spike_kwh", 0.0) + getattr(a, "heat_kwh", 0.0)
          + getattr(a, "climate_cooling_kWh", 0.0) for a in ag_t)
print(f"\nidentity over all {len(ag_t)} homes this hour: fuel-routed {lhs:.3f} kWh  vs  components {rhs:.3f} kWh "
      f"(gap {abs(lhs-rhs):.2e})")
assert np.isclose(lhs, rhs, rtol=1e-6), "the fuel routing does NOT close against the components; do not trust the decomposition"
print("accounting closes ✓. The decomposition below is real bookkeeping, not an assumption")

## Step 1 · measure how big each piece of demand is (one full-year run)

**In plain terms:** we now run the model properly (every home in a sample of
neighbourhoods, simulated for all 8,760 hours of the year) and add up where the energy
goes. We split the yearly total into five pieces:

- **gas baseline**: cooking and hot water,
- **gas heating**: space heating in gas-heated homes,
- **electricity baseline**: always-on appliances, lighting, fridges,
- **electricity people**: the extra electricity from how many people live there,
- **electricity heating**: space heating in electric-heated homes.

This matters because **every parameter feeds exactly one of these pieces**, so once we
know how big each piece is, we know the most a given parameter could possibly move. The
bigger the piece it feeds, the more that parameter can matter. (We re-run the Step 0
balance check on the annual totals here, to be sure nothing drifted over the year.)

*Why a sample and not the whole city?* A representative sample of neighbourhoods gives the
same *shares* as the full city at a fraction of the runtime, and shares are all Step 2
needs. The full-city totals live in the validation notebook (notebook 3).

In [ ]:
gdf = gdf_all[gdf_all.lsoa_code.isin(sorted(gdf_all.lsoa_code.unique())[:SAMPLE_LSOAS])].copy()
m = EnergyModel(gdf=gdf, climate_parquet=CLIM, climate_start=pd.Timestamp(f"{YEAR}-01-01", tz="UTC"),
                local_tz="Europe/London", collect_agent_level=False, config_path=str(CONFIG))
ag = m.household_agents
is_e = np.array([a._resolve_heating_fuel_bucket() == "electric" for a in ag])
tot_e = tot_g = tot_o = base_t = spike_e = heat_e = heat_g = heat_o = cool_t = 0.0
for _ in range(8760):
    m.step()
    for i, a in enumerate(ag):
        tot_e += getattr(a, "electric_kwh", 0.0); tot_g += getattr(a, "gas_kwh", 0.0)
        tot_o += getattr(a, "other_kwh", 0.0)
        base_t += getattr(a, "base_kwh", 0.0)
        spike_e += getattr(a, "spike_kwh", 0.0)
        cool_t += getattr(a, "climate_cooling_kWh", 0.0)
        h = getattr(a, "heat_kwh", 0.0)
        if is_e[i]:   heat_e += h
        elif getattr(a, "_resolve_heating_fuel_bucket", None) and a._resolve_heating_fuel_bucket() == "gas":
            heat_g += h
        else:         heat_o += h

# annual re-check of the Step-0 identity
lhs, rhs = tot_e + tot_g + tot_o, base_t + spike_e + heat_e + heat_g + heat_o + cool_t
print(f"annual accounting: fuel-routed {lhs/1e6:.3f} GWh vs components {rhs/1e6:.3f} GWh (gap {100*abs(lhs-rhs)/lhs:.4f}%)")
assert np.isclose(lhs, rhs, rtol=1e-4), "annual accounting does not close"

comp = {
    "gas baseline":   tot_g - heat_g,
    "gas heating":    heat_g,
    "elec baseline":  tot_e - heat_e - spike_e - cool_t,
    "elec people":    spike_e,
    "elec heating":   heat_e,
}
TOTAL = tot_e + tot_g
print(f"total annual demand over sample: {TOTAL/1e6:.2f} GWh")
sh = pd.Series(comp) / TOTAL * 100
print("\ncomponent shares of total demand:")
print(sh.round(1).astype(str).add(" %").to_string())

## Step 2 · which parameters matter most? (the tornado chart)

**In plain terms:** this is the headline. For each key parameter we nudge it by a
realistic amount and ask *how much does the city's total demand move?* We draw the answers
as a **tornado chart**: one horizontal bar per parameter, longest (most influential) on
top. A long bar means "get this number right or the answer is off"; a short bar means "a
small error here barely shows."

**How the nudges are chosen, and what's an assumption here:**

> **⚙️ Assumption: the size of the nudge.** For the always-on levels and the heating
> sensitivities we nudge by a uniform **±10%**. This puts every parameter on an equal
> footing and answers a *structural* question: **if a number were off by the same amount,
> which piece of demand is big enough for that to matter?** (Step 6 asks a different,
> complementary question, given how *precisely each number was actually measured*, which
> contributes the most real uncertainty, and gets a different, equally useful ranking.
> Neither is "the" answer; together they're the full picture.) The **setpoint** is nudged
> by **+0.5 °C** rather than a percentage, because it works differently (see below).

The setpoint is the one **non-linear** lever: raising the heating "balance point" by half a
degree doesn't scale an existing piece. Rather, it changes *how many hours of the year count as
cold*. So we compute its effect directly from the real weather (how many extra
"degree-hours" of heating that half-degree creates), not by a flat percentage.

In [ ]:
# each entry: (label, component it scales, perturbation as fraction, note)
levers = [
    ("baseline_anchor_gas (+10%)",      "gas baseline",  0.10, "cooking/hot-water level"),
    ("heating_slope_gas (+10%)",        "gas heating",   0.10, "gas heat per degree"),
    ("baseline_anchor_elec (+10%)",     "elec baseline", 0.10, "electricity standing load"),
    ("heating_slope_elec (+10%)",       "elec heating",  0.10, "electric heat per degree"),
    ("per_person_slope (+10%)",         "elec people",   0.10, "per-occupant load"),
]
rows = [(lbl, 100 * frac * comp[c] / TOTAL, note) for lbl, c, frac, note in levers]

# setpoint is non-linear: +0.5C changes the degree-hours. Evaluate on the real climate.
clim = pd.read_parquet(CLIM); clim["timestamp"] = pd.to_datetime(clim["timestamp"], utc=True)
t = clim[(clim.timestamp >= f"{YEAR}-01-01") & (clim.timestamp < f"{YEAR+1}-01-01")].groupby("timestamp")["temp_C"].mean().values
sp = cfg["heating_trigger_temp_C"]
dh0 = np.maximum(0.0, sp - t).sum(); dh1 = np.maximum(0.0, (sp + 0.5) - t).sum()
setpoint_pct = 100 * (dh1/dh0 - 1) * (comp["gas heating"] + comp["elec heating"]) / TOTAL
rows.append(("heating_setpoint (+0.5°C)", setpoint_pct, "non-linear: +0.5°C more degree-hours"))

tor = pd.DataFrame(rows, columns=["parameter", "delta_pct", "note"]).sort_values("delta_pct")
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(tor.parameter, tor.delta_pct, color="#48c")
ax.set_xlabel("change in TOTAL annual demand (%)"); ax.set_title("Parameter influence (tornado)")
for y, (v, n) in enumerate(zip(tor.delta_pct, tor.note)):
    ax.text(v, y, f" {v:.1f}%", va="center", fontsize=9)
plt.tight_layout(); plt.show()
print(tor.round(2).to_string(index=False))
print("\nHeating parameters dominate (heating is the biggest component); the setpoint")
print("is the one non-linear lever. These are the numbers whose accuracy matters most.")

## Step 3 · prove the shortcut is actually correct

**In plain terms:** the tornado in Step 2 was computed with arithmetic, not by re-running
the model. That arithmetic *claims* a +10% nudge to the gas baseline lifts total demand by
exactly 10% of the gas-baseline share. A claim about the model should be checked against
the model, so here we do it the slow, honest way once, as proof.

We take a small set of neighbourhoods and run the full model **twice**: once with the real
settings, and once with a single number (the gas baseline) raised 10%. Then we compare the
change the model *actually* produced (**measured**) with what the Step 2 arithmetic
*predicted*. If they match, the shortcut is trustworthy and every bar in the tornado is
exact. (The cell even stops with an error if they ever disagree, a built-in tripwire in
case the model's internals change in future.)

⏱️ Two ~1-minute runs on a small (4-neighbourhood) subsample.

In [ ]:
VERIFY_LSOAS = sorted(gdf_all.lsoa_code.unique())[:4]
sub = gdf_all[gdf_all.lsoa_code.isin(VERIFY_LSOAS)].copy()

def run_totals(config_path):
    '''Run a year on the subsample; return (total kWh, gas-baseline kWh).'''
    mm = EnergyModel(gdf=sub, climate_parquet=CLIM, climate_start=pd.Timestamp(f"{YEAR}-01-01", tz="UTC"),
                     local_tz="Europe/London", collect_agent_level=False, config_path=str(config_path))
    aa = mm.household_agents
    is_gas = np.array([a._resolve_heating_fuel_bucket() == "gas" for a in aa])
    te = tg = hg = 0.0
    for _ in range(8760):
        mm.step()
        for i, a in enumerate(aa):
            te += getattr(a, "electric_kwh", 0.0); tg += getattr(a, "gas_kwh", 0.0)
            if is_gas[i]: hg += getattr(a, "heat_kwh", 0.0)
    return te + tg, tg - hg

# perturbed config: ONE number changed, +10% on the gas anchor
pert = dict(cfg); pert["baseline_anchor_gas_kwh_per_hour"] = round(cfg["baseline_anchor_gas_kwh_per_hour"] * 1.10, 6)
with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False) as f:
    yaml.safe_dump({"model": pert}, f); PERT = f.name

T0, gasbase0 = run_totals(CONFIG)
T1, _ = run_totals(PERT)
predicted = 100 * 0.10 * gasbase0 / T0          # the tornado's arithmetic, on this subsample
measured  = 100 * (T1 - T0) / T0                 # what the simulator actually did
print(f"gas-baseline share of this subsample's total: {100*gasbase0/T0:.1f}%")
print(f"predicted change from +10% gas anchor : {predicted:+.2f}%")
print(f"measured  change (real perturbed run) : {measured:+.2f}%")
assert abs(predicted - measured) < 0.2, "closed-form and simulator disagree; do not trust the tornado"
print("shortcut verified ✓. The linear levers in the tornado are exact, not approximate")

## Step 4 · why some parameters *don't* move the total

**In plain terms:** you may have noticed the tornado left out a whole family of
parameters: the **multipliers** that make bigger, older, or less-efficient homes use more
and smaller, newer, efficient ones use less. Are we hiding them? No: they genuinely can't
move the *total*, and here's the check that proves it.

These multipliers are built to **average out to 1.0 across all homes**. So turning the
whole set up a notch pushes some homes higher and pulls others lower by the same amount,
like rearranging money between people's pockets without changing how much money there is.
The city total barely budges. What they *do* affect is **which neighbourhood is high or
low**, i.e. the model's spatial accuracy, which is validated separately in notebook 3.
The cell below confirms each multiplier set averages to 1.0 (weighted by how common each
type of home is), so it is a "shape" knob, not a "level" knob.

> **One subtlety, revisited in Step 6.** "Doesn't move the total" is true *at the values we
> fitted*. Whether the total is sensitive to those multipliers being *mis-estimated* is a
> different question: if a whole efficiency or size gradient were systematically off,
> that **would** shift the total. Step 6's global screen measures exactly that, and finds it
> is real (a percent or so); these knobs are level-neutral *by construction* but still
> carry genuine *uncertainty*. We flag both rather than hide the second.

In [ ]:
for key, seg, hf in [("heat_slope_area_bands","floor_area_m2","Gas"),
                     ("sap_band_mult_heating_gas","currentEnergyRating","Gas"),
                     ("building_age_mult_heating_gas","building_age","Gas")]:
    sh_ = L.serl_shares(seg, "Gas", hf); num=den=0.0
    for k,v in cfg[key].items():
        w=float(sh_.get(str(k),0)); num+=w*v; den+=w
    print(f"  {key:32s} stock-weighted mean = {num/den:.3f}  (=> mean-preserving on the total)")
print("\nInterpretation: these are 'shape' knobs, not 'level' knobs. The level is set by")
print("the anchors and slopes above, which is where sensitivity (and scrutiny) belongs.")

## Step 5 · heat pumps: the one assumption that dominates a switching scenario

**In plain terms:** the calibration so far describes homes *as they are today*. But the
model is also used to ask "what if we swapped gas boilers for **heat pumps**?" A heat pump
delivers several units of heat per unit of electricity it draws; that ratio is its
**COP**. To work out the new electricity a converted home needs, the model takes the home's
known gas heat demand and divides by the COP.

That makes the COP the single most important number in any electrification scenario. Unlike
everything else in this notebook, it is **not measured from the smart-meter data**;
it's borrowed from field studies. So it deserves its own spotlight.

> **⚙️ Assumptions: heat pumps.** **COP = 2.8** (a cautious real-world average; well-installed
> systems reach 3.5–4.2) and **boiler efficiency = 0.90** (how much of the gas becomes useful
> heat). Both come from the literature/field data, not SERL. The chart below shows how much a
> converted home's added electricity swings as the COP varies across its plausible range, so
> you can see exactly how much the electrification answer rides on this one borrowed number.

In [ ]:
boiler_eff = cfg.get("boiler_efficiency", 0.90)
# per converted home, added electricity = gas_heat * boiler_eff / COP
gas_heat_per_home = comp["gas heating"] / max(1, (~is_e).sum())   # avg gas-heated home's heat
cops = np.array([2.2, 2.5, 2.8, 3.2, 3.6, 4.0, 4.2])
added = gas_heat_per_home * boiler_eff / cops
ref = gas_heat_per_home * boiler_eff / 2.8
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(cops, added, "o-", color="#1d8a55")
ax.axvline(2.8, color="grey", ls="--", lw=1, label="assumed COP 2.8")
ax.set_xlabel("heat-pump COP"); ax.set_ylabel("added electricity per converted home (kWh/yr)")
ax.set_title("Electrification rides on the COP assumption"); ax.legend(); plt.show()
print(f"assumed COP 2.8 -> {ref:.0f} kWh/yr added per converted home")
print(f"COP 2.2 (poor) -> {gas_heat_per_home*boiler_eff/2.2:.0f}  (+{100*(2.8/2.2-1):.0f}% vs assumed)")
print(f"COP 4.0 (good) -> {gas_heat_per_home*boiler_eff/4.0:.0f}  ({100*(2.8/4.0-1):.0f}% vs assumed)")
print("\n=> A COP error of ±0.6 moves the added electricity per home by ~20%. Because this")
print("is an ASSUMPTION (not a SERL read), it is the electrification result's biggest")
print("single uncertainty and the first thing to pin with field data (heatpumpmonitor.org).")

## Step 6 · a second, independent lens: how much does each parameter's *real* uncertainty matter?

**In plain terms:** Step 2 asked "if a number were off by a round 10%, what moves most?",
which rewards the *biggest* pieces of demand. But we didn't measure every number equally
well: some we pinned down tightly, others loosely. Step 6 asks the complementary question,
**given how precisely each number was *actually* measured, which ones contribute the most
real doubt to the final answer?**, using a completely separate, standard technique (the
*Morris screen*) that re-runs the model many times while jiggling all the parameters
together over their measured uncertainties.

Because it asks a different question, it gives a **different ranking**, and that's the
point: the two together are the full picture. A parameter can be very influential yet add
little doubt if we measured it precisely (the gas heating rate is exactly this), while a
loosely-measured one matters more than its raw influence suggests. Morris reports two
numbers per parameter:

- **μ\*** is the mean absolute elementary effect: the knob's overall influence on total
  demand (the screening ranking).
- **σ** is the spread of the elementary effects: **non-linearity / interaction** with the
  other knobs. σ ≈ 0 means the effect is the same everywhere in parameter space, i.e.
  the linear closed-form is exact.

Morris does two jobs here. First, the decisive cross-check: **σ/μ\* is tiny for every
parameter**, which means each parameter's effect is the same no matter what the others are
doing. The model is **near-linear**, with no hidden interactions. That is what guarantees
the Step 2 tornado's arithmetic is *exact for any nudge size*, not a lucky approximation
(Step 3 already proved it for one lever; this proves it for all of them). Both lenses also
agree that **heating is central**: heating levers sit at or near the top of each, even
though the exact order differs by construction (Step 2 weights by size, Step 6 by measured
precision). Second, Morris **extends** the tornado: because it jiggles *whole gradients*
(all the efficiency/age/size bands together), it surfaces the one thing the tornado set
aside. A *systematic* mis-estimate of an entire gradient would move the total ~1–2%. Those
gradients are level-neutral at their fitted values (Step 4) yet still carry real
uncertainty. Read together: nothing behaves non-linearly, heating dominates, and the
gradient multipliers, invisible to the tornado, earn a line in the paper's uncertainty
budget.

> **⚙️ Notes on this run.** The measurement uncertainties (how far each parameter is
> nudged) come from the SERL data's own sampling error. It is run on a representative
> **subsample** of neighbourhoods, enough to settle the ranking and the interaction check,
> which is all we need here; a full-resolution version (more paths, more neighbourhoods) is
> available for the final paper but changes only the decimal places, not the conclusion.

*(This cell runs the Morris screen inline via `sa_morris.main()` on the current config, then
caches the result to `results/sensitivity_analysis/sa_morris_newcastle_v2.csv`; set
`FORCE_SA = True` to re-run it from scratch. These are real full-year model runs, so a fresh
run takes a few minutes.)*

In [ ]:
FORCE_SA = False                 # True -> re-run the screen even if the CSV exists
SA_R, SA_NLSOA = 4, 7            # Morris trajectories / stratified LSOA subsample; bump (e.g. 8, 18) for the full-resolution paper run
PARAM_TABLE = REPO / "results/sensitivity_analysis/sa_param_table_v2.yaml"
MORRIS      = REPO / "results/sensitivity_analysis/sa_morris_newcastle_v2.csv"
# Run the screen inline so the notebook reproduces it end-to-end (real r*(k+1) full-year model runs).
if FORCE_SA or not PARAM_TABLE.exists():
    import build_sa_param_table_v2 as _sapt; _sapt.main()   # param ranges: centrals from the current config, SEs from the SERL bootstrap
if FORCE_SA or not MORRIS.exists():
    import sa_morris
    sa_morris.main(["--calib-dir", str(REPO / "results/serl_ledger"),
                    "--param-table", str(PARAM_TABLE), "--city", "newcastle", "--year", "2023",
                    "--r", str(SA_R), "--levels", "4", "--n-lsoa", str(SA_NLSOA),
                    "--max-procs", "6", "--label", "newcastle_v2"])
if True:
    mo = pd.read_csv(MORRIS).sort_values("mu_star_energy_pct_base", ascending=False)
    mo["sigma_over_mustar"] = mo["sigma_energy"] / mo["mu_star_energy"].replace(0, np.nan)
    # knobs the closed-form tornado models vs the mean-preserving multipliers it omits
    TORNADO_KNOBS = {"heating_setpoint_C", "heating_slope_kWh_per_deg",
                     "baseline_anchor_gas_kwh_per_hour", "baseline_anchor_elec_kwh_per_hour",
                     "setpoint_setback_C"}
    mo["in_tornado"] = mo["knob"].isin(TORNADO_KNOBS)
    # gradient-multiplier knobs: level-neutral at fitted values, but their systematic
    # (whole-gradient) uncertainty is what Morris screens here
    GRADIENTS = {"heat_slope_area_bands", "sap_band_mult_heating_gas",
                 "building_age_mult_heating_gas", "baseline_elec_area_bands"}
    mo["kind"] = np.where(mo["knob"].isin(TORNADO_KNOBS), "level lever (in tornado)",
                          np.where(mo["knob"].isin(GRADIENTS), "gradient (uncertainty only)", "other"))
    cmap = {"level lever (in tornado)": "#48c", "gradient (uncertainty only)": "#c9a227", "other": "#999"}
    colours = [cmap[k] for k in mo["kind"]]

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
    a1.barh(mo["knob"][::-1], mo["mu_star_energy_pct_base"][::-1], color=colours[::-1])
    a1.set_xlabel("μ*  (% of total demand per ±2 SE move)")
    a1.set_title("Morris influence\n(blue = level lever the tornado models; gold = gradient, uncertainty only)", fontsize=9)
    a2.scatter(mo["mu_star_energy_pct_base"], mo["sigma_over_mustar"], color=colours, s=45)
    a2.axhline(0.15, color="grey", ls="--", lw=1)
    a2.set_xlabel("μ*  (% of total)"); a2.set_ylabel("σ / μ*  (non-linearity / interaction)")
    a2.set_title("every point far below 0.15 ⇒ near-linear ⇒ closed-form is exact", fontsize=9)
    plt.tight_layout(); plt.show()

    print(mo[["knob", "mu_star_energy_pct_base", "sigma_over_mustar", "kind"]].round(3).to_string(index=False))
    top_morris = mo["knob"].iloc[0]
    top_tor = tor.sort_values("delta_pct", ascending=False)["parameter"].iloc[0]
    worst_nl = mo["sigma_over_mustar"].max()
    grad_max = mo.loc[mo["knob"].isin(GRADIENTS), "mu_star_energy_pct_base"].max()
    print(f"\nNEAR-LINEAR (the decisive cross-check): worst σ/μ* = {worst_nl:.2f} "
          f"({'✓ <0.15' if worst_nl < 0.15 else 'REVIEW'}) => no interactions; the Step 2 tornado's")
    print("  arithmetic is exact for any nudge size, not just the one lever checked in Step 3.")
    print(f"DIFFERENT LENS: Morris tops with {top_morris} (measured-uncertainty basis); the tornado")
    print(f"  tops with {top_tor} (uniform-10% basis). Both put HEATING on top; the order differs")
    print("  because a very precisely-measured lever (the gas heating rate) adds little real doubt.")
    print(f"EXTENDS: the efficiency/age/size gradients reach μ* ≈ {grad_max:.1f}% of total, level-neutral")
    print("  at their fitted values (Step 4) but a real systematic uncertainty the tornado sets aside;")
    print("  the one thing to add to a full uncertainty budget for the paper.")

## Summary

- **The decomposition is checked, not assumed:** the per-home fuel routing closes
  against the base/people/heating components hourly and annually (Step 0, Step 1),
  and the closed-form tornado is verified once against a real perturbed run (Step 3).
- **Heating parameters (slopes + setpoint) dominate** the total, because heating is
  the largest component; their accuracy matters most for the headline demand.
- **Multipliers are shape, not level:** mean-preserving, so they affect *where*
  demand is, not *how much* in total.
- **A Morris screen is the second lens** (Step 6): σ/μ\* is tiny everywhere (the model is
  near-linear → the tornado's analytic reads are exact), and both methods put heating on
  top, though the exact order differs *by design* (the tornado weights by demand size, Morris
  by each parameter's measured precision). Morris also reveals what the tornado sets aside:
  a *systematic* mis-estimate of a whole efficiency/age/size gradient would move the total
  ~1–2%, level-neutral at the fitted values, but a real uncertainty for the budget. The
  two lenses together are the complete SA.
- **For electrification specifically, the COP is the lever:** an assumption, not a
  SERL read, and the largest single uncertainty in a conversion scenario. Everything
  else electrification needs (the gas heat demand it converts) is SERL-calibrated and
  validated (notebook 3).

This notebook is the paper's sensitivity analysis end-to-end: a transparent analytic
tornado, verified against the simulator, and confirmed by a global Morris screen.